# A3 — Speech model comparison for Basira (Listen stage)

**Purpose:** choose the acoustic model for per-phoneme scoring (issue #3). Compares two Apache-2.0 models on the
contrasts Basira depends on: Kuwaiti dialect variants (ق→[g], ج→[j]) and Arabic error categories (dot and emphatic
confusions).

| Model | Output | Licence |
| --- | --- | --- |
| `facebook/wav2vec2-xlsr-53-espeak-cv-ft` | phonemes (espeak IPA) | Apache-2.0 |
| `jonatasgrosman/wav2vec2-large-xlsr-53-arabic` | Arabic letters | Apache-2.0 |

**Audio sources.** (1) SYNTHETIC: Google TTS Arabic voice (`gTTS`), used only for this internal comparison, never
shipped. (2) OPTIONAL REAL ADULT RECORDINGS: short voice notes uploaded by a consenting team member, for the Kuwaiti
dialect contrasts no TTS voice can produce here. Neither is child speech; results show whether a model *can* represent
a contrast, not how it performs on children.

**How to run:** Runtime → Change runtime type → T4 GPU (optional, CPU also works) → Runtime → Run all.
Then copy the block printed by the last cell and paste it back to Claude.

In [ ]:
%%capture
!apt-get -qq install -y espeak-ng > /dev/null
!pip -q install -U gTTS phonemizer==3.3.0
# transformers, torch, librosa, soundfile: use the versions preinstalled in Colab

In [ ]:
import json, os, shutil, glob, numpy as np, torch, librosa
from gtts import gTTS
from transformers import AutoProcessor, AutoModelForCTC
from phonemizer.backend import EspeakBackend
from phonemizer.separator import Separator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
shutil.rmtree("audio", ignore_errors=True); os.makedirs("audio")  # never reuse stale/partial clips
HARAKAT = "".join(chr(c) for c in range(0x064B, 0x0653)) + "ٰ"
strip = lambda s: "".join(ch for ch in s if ch not in HARAKAT)

## Test cases

Each case synthesizes `spoken` with a TTS voice and scores it against the **expected** (correct MSA) word.
`hypothesis` is what a good model should show at the target position.

The Kuwaiti ق→[g] contrast cannot be produced by the Google voice, so it comes from the optional recording step
below. ج→[j] is tested by synthesizing `دَيَاي` (the text itself contains [j]).

In [ ]:
CASES = [
  # pairs = [(expected phone, competing phone), ...] compared at the expected phone's aligned frames
  dict(id="qaaf_msa",      expected="قَلَم",  spoken="قَلَم",  letter="ق", pairs=[("q","ɡ"),("q","ʔ")], hyp="correct: q"),
  dict(id="thaa_msa",      expected="ثَمِيك", spoken="ثَمِيك", letter="ث", pairs=[("θ","t"),("θ","f")], hyp="correct: θ (pseudoword)"),
  dict(id="thaa_as_taa",   expected="ثَمِيك", spoken="تَمِيك", letter="ث", pairs=[("θ","t")], hyp="dot confusion: hears t (pseudoword)"),
  dict(id="dhaal_msa",     expected="ذَرُوم", spoken="ذَرُوم", letter="ذ", pairs=[("ð","d")], hyp="correct: ð (pseudoword)"),
  dict(id="dhaal_as_daal", expected="ذَرُوم", spoken="دَرُوم", letter="ذ", pairs=[("ð","d")], hyp="dot confusion: hears d (pseudoword)"),
  dict(id="khaa_msa",      expected="خَلُوم", spoken="خَلُوم", letter="خ", pairs=[("χ","ħ"),("χ","x")], hyp="correct: χ (pseudoword)"),
  dict(id="khaa_as_hhaa",  expected="خَلُوم", spoken="حَلُوم", letter="خ", pairs=[("χ","ħ")], hyp="dot confusion: hears ħ (pseudoword)"),
  dict(id="saad_msa",      expected="صَمُوك", spoken="صَمُوك", letter="ص", pairs=[("s̪","s"),("a.","a")], hyp="correct: emphatic (pseudoword)"),
  dict(id="saad_as_siin",  expected="صَمُوك", spoken="سَمُوك", letter="ص", pairs=[("s̪","s"),("a.","a")], hyp="emphatic confusion: hears plain s / plain a (pseudoword)"),
  dict(id="jiim_msa",      expected="دَجَاج", spoken="دَجَاج", letter="ج", pairs=[("dʒ","j")], hyp="correct: dʒ"),
  dict(id="jiim_as_y",     expected="دَجَاج", spoken="دَيَاي", letter="ج", pairs=[("dʒ","j")], hyp="dialect: hears j, not dʒ"),
]
for c in CASES:
    c["path"] = f"audio/{c['id']}.mp3"
    gTTS(c["spoken"], lang="ar").save(c["path"])
    assert os.path.getsize(c["path"]) > 1000, f"empty clip for {c['id']}"
    c["source"] = "SYNTHETIC gTTS-ar"
print("synthesized", len(CASES), "clips")

## Optional: your own voice notes (real Kuwaiti speech)

This is the only way to test the headline feature (ق said as [g]). Record **4 short voice notes** on your phone
(WhatsApp or the voice recorder app is fine), each saying ONE word clearly, then upload them when the next cell asks:

| File name (start with) | Say | How |
| --- | --- | --- |
| `qalam_q` | قَلَم | MSA: ق as **[q]** — "qalam" |
| `qalam_g` | قَلَم | ق as **[g]** (Kuwaiti variant) — "galam" |
| `qalam_hamza` | قَلَم | ق as **[ʔ]** (Cairene, should NOT count as Kuwaiti) — "ʔalam" |
| `dajaj_y` | دَجَاج | Gulf: "dayay" |

Any format works (m4a, mp3, ogg, wav). Click **Cancel upload** to skip this step; the rest still runs.
These recordings stay in this Colab session only and are deleted when the runtime ends.

In [ ]:
SPEAKER = "adult Egyptian team member producing the target sounds (not a Kuwaiti speaker)"
REC = [
  dict(id="rec_qalam_q",     expected="قَلَم",  letter="ق", pairs=[("q","ɡ"),("q","ʔ")], hyp="REAL MSA [q]: hears q"),
  dict(id="rec_qalam_g",     expected="قَلَم",  letter="ق", pairs=[("q","ɡ"),("q","ʔ")], hyp="REAL [g] (Kuwaiti variant): hears g, not q"),
  dict(id="rec_qalam_hamza", expected="قَلَم",  letter="ق", pairs=[("q","ɡ"),("q","ʔ")], hyp="REAL [ʔ] (Cairene, NOT Kuwaiti): hears ʔ, not g"),
  dict(id="rec_dajaj_y",     expected="دَجَاج", letter="ج", pairs=[("dʒ","j")], hyp="REAL [j] (Kuwaiti variant): hears j, not dʒ"),
]
try:
    from google.colab import files
    uploaded = files.upload()
except Exception as e:
    uploaded = {}
    print("upload skipped:", type(e).__name__)
for r in REC:
    key = r["id"].replace("rec_", "")
    match = [n for n in uploaded if n.lower().startswith(key)]
    if match:
        path = f"audio/{r['id']}_{match[0]}"
        with open(path, "wb") as f: f.write(uploaded[match[0]])
        CASES.append(dict(r, spoken="(recording)", path=path, source="REAL voice note: " + SPEAKER))
        print("added", r["id"], "from", match[0])
    else:
        print("no file for", key, "- skipped")

## Models and CTC forced alignment

In [ ]:
def load(name):
    proc = AutoProcessor.from_pretrained(name)
    model = AutoModelForCTC.from_pretrained(name).to(DEVICE).eval()
    vocab = proc.tokenizer.get_vocab()
    return proc, model, vocab

def log_posteriors(proc, model, path):
    wav, _ = librosa.load(path, sr=16000)
    inp = proc(wav, sampling_rate=16000, return_tensors="pt").input_values.to(DEVICE)
    with torch.no_grad():
        logits = model(inp).logits[0]
    return torch.log_softmax(logits, -1).cpu().numpy()      # (T, V)

def ctc_align(lp, targets, blank):
    # Viterbi CTC forced alignment. Returns list of (start, end) frame spans per target token.
    T, L = lp.shape[0], len(targets)
    ext = [blank]
    for t in targets: ext += [t, blank]
    S = len(ext); NEG = -1e9
    dp = np.full((T, S), NEG); bp = np.zeros((T, S), dtype=int)
    dp[0, 0] = lp[0, ext[0]]
    if S > 1: dp[0, 1] = lp[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            cands = [(dp[t-1, s], s)]
            if s >= 1: cands.append((dp[t-1, s-1], s-1))
            if s >= 2 and ext[s] != blank and ext[s] != ext[s-2]: cands.append((dp[t-1, s-2], s-2))
            best, arg = max(cands)
            dp[t, s] = best + lp[t, ext[s]]; bp[t, s] = arg
    s = S-1 if S < 2 or dp[T-1, S-1] >= dp[T-1, S-2] else S-2
    path = []
    for t in range(T-1, -1, -1):
        path.append(s); s = bp[t, s]
    path.reverse()
    spans = [[None, None] for _ in range(L)]
    for t, s in enumerate(path):
        if s % 2 == 1:
            i = s // 2
            if spans[i][0] is None: spans[i][0] = t
            spans[i][1] = t + 1
    return spans

def segment_scores(lp, span, ids):
    a, b = span
    return {tok: float(lp[a:b, i].mean()) for tok, i in ids.items()}

def greedy(lp, inv, blank):
    best = lp.argmax(-1); out = []; prev = None
    for k in best:
        if k != prev and k != blank: out.append(inv[k])
        prev = k
    return out

In [ ]:
# ---- phoneme model ----
PH = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"
ph_proc, ph_model, ph_vocab = load(PH)
ph_inv = {v: k for k, v in ph_vocab.items()}
ph_blank = ph_proc.tokenizer.pad_token_id
espeak = EspeakBackend("ar", preserve_punctuation=False, with_stress=False)
SEP = Separator(phone=" ", word=" | ", syllable="")

def to_ph_tokens(text):
    raw = espeak.phonemize([text], separator=SEP, strip=True)[0]
    toks, unknown = [], []
    for p in raw.replace("|", " ").split():
        if p in ph_vocab: toks.append(p); continue
        # greedy longest-match split for multi-char espeak symbols
        i = 0
        while i < len(p):
            for j in range(len(p), i, -1):
                if p[i:j] in ph_vocab: toks.append(p[i:j]); i = j; break
            else:
                unknown.append(p[i]); i += 1
    return raw, toks, unknown

# ---- letter model ----
GR = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
gr_proc, gr_model, gr_vocab = load(GR)
gr_inv = {v: k for k, v in gr_vocab.items()}
gr_blank = gr_proc.tokenizer.pad_token_id
print("loaded both models")

In [ ]:
results = []
for c in CASES:
    r = dict(id=c["id"], hyp=c["hyp"], expected=c["expected"], spoken=c["spoken"], source=c["source"])

    # phoneme model: align expected espeak phones; compare expected vs alternative phone at the target
    raw, toks, unk = to_ph_tokens(c["expected"])
    lp = log_posteriors(ph_proc, ph_model, c["path"])
    r["ph_expected_raw"] = raw; r["ph_unknown"] = unk
    r["ph_heard_free"] = " ".join(greedy(lp, ph_inv, ph_blank))
    r["ph_pairs"] = []
    spans = ctc_align(lp, [ph_vocab[t] for t in toks], ph_blank) if toks else []
    for exp_t, alt_t in c["pairs"]:
        if exp_t not in toks or alt_t not in ph_vocab:
            r["ph_pairs"].append(f"{exp_t}/{alt_t}: n/a (expected phones {toks})"); continue
        k = toks.index(exp_t)
        sc = segment_scores(lp, spans[k], {exp_t: ph_vocab[exp_t], alt_t: ph_vocab[alt_t]})
        win = exp_t if sc[exp_t] >= sc[alt_t] else alt_t
        r["ph_pairs"].append(f"{exp_t} {sc[exp_t]:.2f} vs {alt_t} {sc[alt_t]:.2f} -> {win}")
    # letter model: align expected letters; report top letters at the target position
    letters = [ch for ch in strip(c["expected"]) if ch in gr_vocab]
    lp2 = log_posteriors(gr_proc, gr_model, c["path"])
    spans2 = ctc_align(lp2, [gr_vocab[ch] for ch in letters], gr_blank)
    k2 = letters.index(c["letter"])
    a, b = spans2[k2]
    mean = lp2[a:b].mean(0)
    top = [gr_inv[i] for i in np.argsort(-mean)[:3] if i != gr_blank]
    r["gr_heard_free"] = "".join(greedy(lp2, gr_inv, gr_blank))
    r["gr_top_at_target"] = top
    results.append(r)
print("scored", len(results), "cases")

In [ ]:
# ===== COPY EVERYTHING BELOW THIS LINE AND PASTE IT TO CLAUDE =====
print("A3-MODEL-COMPARISON v4")
import transformers; print("python", __import__("sys").version.split()[0], "| transformers", transformers.__version__, "| torch", torch.__version__)
print("device", DEVICE)
for r in results:
    print("---", r["id"], "|", r["hyp"])
    print("  expected", r["expected"], "spoken", r["spoken"], "|", r["source"])
    print("  PH espeak:", r["ph_expected_raw"], "| unknown:", r["ph_unknown"])
    print("  PH heard :", r["ph_heard_free"])
    print("  PH at target:", " | ".join(r["ph_pairs"]))
    print("  GR heard :", r["gr_heard_free"], "| top letters at target:", r["gr_top_at_target"])
json.dump(results, open("a3_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)